# 04.06 · Human-in-the-loop for agents

**Module 04 · Middleware, Guardrails & Human-in-the-Loop** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 04.01–04.05; Lesson 03.02 (checkpointers). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Pause** an agent before a sensitive tool runs with `HumanInTheLoopMiddleware`, and read the pause (`version="v2"` → `.interrupts`).
2. **Resume** with each decision type: **approve**, **edit**, **reject**, **respond**, and with several decisions at once.
3. **Configure** policies per tool: allowed decisions, descriptions, and conditional `when=` rules.
4. **Explain** why HITL needs a checkpointer, and make a pause **survive a kernel restart** (SQLite).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) | **Main page.** `interrupt_on`, decision types, responding to interrupts |
| [Interrupts (LangGraph)](https://docs.langchain.com/oss/python/langgraph/interrupts) | How a pause is stored and resumed with `Command(resume=...)` |
| [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | Checkpointers and threads (recap of 03.02) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph", "langgraph-checkpoint-sqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/04-middleware-guardrails-hitl/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# Step 5: The inbox and a send tool that only writes a file to a git-ignored "outbox" (nothing is ever really sent).
import json
import shutil

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from mastery.testing import ScriptedChatModel

INBOX = json.loads((REPO_ROOT / "data" / "samples" / "inbox" / "inbox.json").read_text(encoding="utf-8"))
EMAILS = {e["id"]: e for e in INBOX["emails"]}
OUTBOX = REPO_ROOT / "data" / "outputs" / "m04" / "04-06-outbox"
shutil.rmtree(OUTBOX, ignore_errors=True)
OUTBOX.mkdir(parents=True)


@tool
def send_reply(email_id: str, body: str) -> str:
    """Send a reply to the sender of one email."""
    (OUTBOX / f"reply-{email_id}.txt").write_text(body, encoding="utf-8")
    return f"Sent reply to {email_id}."


@tool
def ask_user(question: str) -> str:
    """Ask the human a question. A person answers it; this function never runs."""
    return "(no answer)"


def outbox() -> list[str]:
    return sorted(p.name for p in OUTBOX.iterdir())


def proposes(*calls):
    """A scripted model: first proposes these tool calls, then summarizes the last tool result."""
    first = AIMessage("", tool_calls=[{"name": n, "args": a, "id": f"c{i}"} for i, (n, a) in enumerate(calls, 1)])
    return ScriptedChatModel(responses=[first, lambda msgs: f"Done. Last tool said: {msgs[-1].content}"])


print("outbox:", outbox())

**You should now see** `outbox: []`.

---
## Part 1 · Pause before a sensitive tool

`HumanInTheLoopMiddleware` runs **after the model** proposes tool calls and **before** any of them run. For each tool you list in `interrupt_on`, it stops the run and hands you an **interrupt**: what the agent wants to do, and which decisions you may take.

| `interrupt_on` value | Meaning |
|---|---|
| `True` | Pause; all decisions allowed (approve, edit, reject, respond) |
| `False` (or not listed) | Never pause; the tool runs immediately |
| `{"allowed_decisions": [...], "description": ..., "when": ...}` | Pause with a custom policy (Part 3) |

A pause means "save everything and stop", so the agent **needs a checkpointer** and a `thread_id` (Lesson 03.02) to continue later. Passing `version="v2"` to `invoke` returns an object with `.value` (the state) and `.interrupts` (the pauses).

In [ ]:
# 🧪 Offline: the agent proposes a reply. It pauses; nothing is sent.
def make_agent(model, policy=None, checkpointer=None):
    return create_agent(model, tools=[send_reply, ask_user],
                        middleware=[HumanInTheLoopMiddleware(interrupt_on=policy or {"send_reply": True})],
                        checkpointer=checkpointer if checkpointer is not None else InMemorySaver())


agent = make_agent(proposes(("send_reply", {"email_id": "e-103", "body": "Thanks, the delivery is on its way."})))
cfg = {"configurable": {"thread_id": "approve-demo"}}
result = agent.invoke({"messages": [HumanMessage("Reply to e-103.")]}, cfg, version="v2")

pause = result.interrupts[0]
print("paused:", bool(result.interrupts), "| outbox:", outbox())
print(json.dumps(pause.value, indent=2))

**You should now see** `paused: True | outbox: []` and the interrupt value:

- `action_requests`: one entry per paused call, with `name`, `args` and a `description` (`Tool execution requires approval …`).
- `review_configs`: for each tool, `allowed_decisions`, here `["approve", "edit", "reject", "respond"]`.

This is exactly what a review screen would show a person. In a web app you send `pause.value` to the UI and wait; here we answer in code.

---
## Part 2 · The four decisions

You resume by invoking again on the **same thread** with `Command(resume={"decisions": [...]})`, one decision per paused call, **in the same order** as `action_requests`.

| Decision | Shape | What happens |
|---|---|---|
| approve | `{"type": "approve"}` | The tool runs with the proposed arguments |
| edit | `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` | The tool runs with **your** arguments; the model is told they were edited |
| reject | `{"type": "reject", "message": "why"}` | The tool does **not** run; the model receives your reason |
| respond | `{"type": "respond", "message": "answer"}` | The tool does not run; your text **is** the tool result (good for `ask_user`) |

In [ ]:
# 🧪 Offline: approve. The tool runs, and the reply lands in the outbox.
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg, version="v2")
print("interrupts:", result.interrupts, "| outbox:", outbox())
print("final:", result.value["messages"][-1].content)

**You should now see** `interrupts: () | outbox: ['reply-e-103.txt']` and `final: Done. Last tool said: Sent reply to e-103.`

In [ ]:
# 🧪 Offline: edit (fix the body), reject (with a reason), respond (answer a question). Each on its own thread.
def run(model, decision, thread, policy=None):
    a = make_agent(model, policy)
    c = {"configurable": {"thread_id": thread}}
    a.invoke({"messages": [HumanMessage("Handle the email.")]}, c, version="v2")
    return a.invoke(Command(resume={"decisions": [decision]}), c, version="v2").value["messages"]


edited = run(proposes(("send_reply", {"email_id": "e-101", "body": "Refund of COP 960,000 approved!"})),
             {"type": "edit", "edited_action": {"name": "send_reply", "args": {
                 "email_id": "e-101", "body": "We are checking the double charge and will reply within one day."}}},
             "edit-demo")
print("edit    →", (OUTBOX / "reply-e-101.txt").read_text(encoding="utf-8"))

rejected = run(proposes(("send_reply", {"email_id": "e-104", "body": "Sure, here is the password."})),
               {"type": "reject", "message": "Never share credentials. Escalate to security."}, "reject-demo")
print("reject  →", [m.content for m in rejected if isinstance(m, ToolMessage)][-1])

answered = run(proposes(("ask_user", {"question": "Formal or friendly tone?"})),
               {"type": "respond", "message": "Formal, please."}, "respond-demo",
               policy={"ask_user": {"allowed_decisions": ["respond"]}})
print("respond →", [m.content for m in answered if isinstance(m, ToolMessage)][-1])
print("outbox:", outbox())

**You should now see**

- `edit    → We are checking the double charge and will reply within one day.` (your text, not the model's).
- `reject  → User rejected the tool call for `send_reply` with reason: Never share credentials. Escalate to security.`
- `respond → Formal, please.`
- `outbox: ['reply-e-101.txt', 'reply-e-103.txt']`: nothing for e-104.

### What just happened?

Every run followed the same cycle: **model proposes → middleware pauses → you decide → the agent continues**.

1. `invoke(..., version="v2")` ran until the model proposed `send_reply`; `HumanInTheLoopMiddleware` stopped **before** the tool, and the checkpointer saved the state under the `thread_id`.
2. `Command(resume={"decisions": [...]})` on the **same** `thread_id` loaded that state and applied your decision.
3. The tool ran (approve, edit) or didn't (reject, respond); either way the model got a `ToolMessage` and wrote its final answer.

After an edit, the middleware also adds a short note to the conversation so the model knows its call was changed. Look at `edited` if you're curious.

---
## Part 3 · Policies: several calls, allowed decisions, `when=`

One model turn can propose **several** tool calls. The middleware pauses **once** with all of them; you send one decision per call, in order. Two rules are enforced, and a wrong resume is a `ValueError`:

- the number of decisions must equal the number of paused calls;
- each decision must be in that tool's `allowed_decisions`.

> ⚠️ **Check decisions before you resume.** In our tests (langgraph 1.2.14), after a resume failed with the wrong number of decisions, resending corrected decisions **on the same thread** still failed with the same error. Validate first (the helper below), and if a thread gets stuck, rerun the request on a **new** `thread_id`.

In [ ]:
# 🧪 Offline: two replies in one turn; approve one, reject the other. A helper validates before resuming.
def check_decisions(pause_value: dict, decisions: list[dict]) -> None:
    """Raise a clear error *before* resuming if the decisions don't fit the pause."""
    requests, configs = pause_value["action_requests"], pause_value["review_configs"]
    if len(decisions) != len(requests):
        raise ValueError(f"{len(requests)} call(s) paused but {len(decisions)} decision(s) given.")
    allowed = {c["action_name"]: c["allowed_decisions"] for c in configs}
    for req, dec in zip(requests, decisions):
        if dec["type"] not in allowed[req["name"]]:
            raise ValueError(f"'{dec['type']}' is not allowed for {req['name']}; use one of {allowed[req['name']]}.")


agent = make_agent(proposes(("send_reply", {"email_id": "e-105", "body": "Confirmed for Friday."}),
                            ("send_reply", {"email_id": "e-107", "body": "Forwarding everything now."})),
                   policy={"send_reply": {"allowed_decisions": ["approve", "reject"]}})
cfg = {"configurable": {"thread_id": "multi-demo"}}
pause = agent.invoke({"messages": [HumanMessage("Reply to e-105 and e-107.")]}, cfg, version="v2").interrupts[0]
print("paused calls:", [r["args"]["email_id"] for r in pause.value["action_requests"]])

for attempt in ([{"type": "approve"}],
                [{"type": "edit", "edited_action": {}}, {"type": "approve"}],
                [{"type": "approve"}, {"type": "reject", "message": "e-107 is phishing."}]):
    try:
        check_decisions(pause.value, attempt)
    except ValueError as e:
        print("❌", e)
        continue
    agent.invoke(Command(resume={"decisions": attempt}), cfg)
    print("✅ resumed | outbox:", outbox())

**You should now see**

```
paused calls: ['e-105', 'e-107']
❌ 2 call(s) paused but 1 decision(s) given.
❌ 'edit' is not allowed for send_reply; use one of ['approve', 'reject'].
✅ resumed | outbox: ['reply-e-101.txt', 'reply-e-103.txt', 'reply-e-105.txt']
```

**Conditional pauses.** Asking a human about *every* call trains people to click "approve" without reading. `when=` (needs `langchain>=1.3.3`) takes the proposed call and returns `True` to pause; `description=` can be a function that writes a helpful review text.

In [ ]:
# 🧪 Offline: only pause when the reply mentions money; the description explains why.
import re

MONEY = re.compile(r"\b(COP|USD|refund|reembolso)\b|\$\s?\d", re.I)

policy = {"send_reply": {
    "allowed_decisions": ["approve", "edit", "reject"],
    "when": lambda request: bool(MONEY.search(request.tool_call["args"].get("body", ""))),
    "description": lambda tool_call, state, runtime: f"Money mentioned in reply to {tool_call['args']['email_id']}: check the amount.",
}}
for email_id, body in [("e-108", "Thanks, see you at the meetup."), ("e-109", "We'll refund USD 120 today.")]:
    a = make_agent(proposes(("send_reply", {"email_id": email_id, "body": body})), policy)
    r = a.invoke({"messages": [HumanMessage("Reply.")]}, {"configurable": {"thread_id": f"when-{email_id}"}}, version="v2")
    print(email_id, "→", r.interrupts[0].value["action_requests"][0]["description"] if r.interrupts else "sent without review")

**You should now see**

```
e-108 → sent without review
e-109 → Money mentioned in reply to e-109: check the amount.
```

---
## Part 4 · Why a checkpointer, and a pause that survives a restart

A pause can last minutes or days (a person is on lunch). The paused run must live **outside** your Python variables. That is the checkpointer's job: it stores the state *and* the pending interrupt under the `thread_id`.

In [ ]:
# 🧪 Offline: without a checkpointer the agent still pauses, but you can never resume.
no_memory = create_agent(proposes(("send_reply", {"email_id": "e-103", "body": "Hi"})), tools=[send_reply],
                         middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_reply": True})])
first = no_memory.invoke({"messages": [HumanMessage("Reply to e-103.")]})
print("paused:", "__interrupt__" in first)
try:
    no_memory.invoke(Command(resume={"decisions": [{"type": "approve"}]}))
except RuntimeError as e:
    print("💥", e)

**You should now see** `paused: True` and `💥 Cannot use Command(resume=...) without checkpointer`. (Without `version="v2"`, the pause shows up as an `__interrupt__` key in the result dict.)

Now the realistic case: a **SQLite** checkpointer (Lesson 03.02). We pause, throw the agent away (as a kernel restart would), build a **new** agent on the same file, and resume.

In [ ]:
# 🧪 Offline: pause → "restart" → resume from SQLite.
from mastery.memory import make_checkpointer

DB = REPO_ROOT / "data" / "outputs" / "m04" / "04-06-hitl.sqlite"
DB.unlink(missing_ok=True)
cfg = {"configurable": {"thread_id": "overnight"}}

before = make_agent(proposes(("send_reply", {"email_id": "e-110", "body": "Your invoice is attached."})),
                    checkpointer=make_checkpointer("sqlite", DB))
before.invoke({"messages": [HumanMessage("Reply to e-110.")]}, cfg)
del before                                   # Everything in memory is gone now.

after = make_agent(proposes(), checkpointer=make_checkpointer("sqlite", DB))
waiting = after.get_state(cfg).interrupts
print("still waiting:", [r["args"]["email_id"] for r in waiting[0].value["action_requests"]])
after.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
print("outbox:", outbox())

**You should now see** `still waiting: ['e-110']` and `reply-e-110.txt` in the outbox (next to the earlier files, including `reply-e-108.txt`, which Part 3 sent without review). The new agent's scripted model never proposed anything (`proposes()` with no calls): the pending call came from the **file**. To try a real restart: run the cell up to `del before`, use **Kernel → Restart**, rerun Steps 1–5 and the second half.

`get_state(cfg).interrupts` is how a review queue finds work: list your open threads and show each pending interrupt.

---
## Part 5 · 💸 Live: a real model proposes, you decide

> 💸 Uses a few hundred tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: gpt-oss-20b drafts a reply to e-102; we reject it once with feedback, then approve the second try.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live = make_agent(get_model("fast"), policy={"send_reply": {"allowed_decisions": ["approve", "edit", "reject"]}})
    cfg = {"configurable": {"thread_id": "live-demo"}}
    e = EMAILS["e-102"]
    r = live.invoke({"messages": [HumanMessage(f"Reply to email e-102 using send_reply.\n\nFrom: {e['from']}\n{e['body']}")]},
                    cfg, version="v2")
    for decision in ({"type": "reject", "message": "Too long. Two sentences, and sign as 'Support team'."}, {"type": "approve"}):
        if not r.interrupts:
            print("No pause: the model answered without calling send_reply:", r.value["messages"][-1].content[:200])
            break
        print("proposed body:", r.interrupts[0].value["action_requests"][0]["args"].get("body", "")[:300], "\n→", decision["type"])
        r = live.invoke(Command(resume={"decisions": [decision] * len(r.interrupts[0].value["action_requests"])}), cfg,
                        version="v2")
    print("outbox:", outbox())
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) a first proposed body, `→ reject`, a shorter second body signed "Support team", `→ approve`, and `reply-e-102.txt` in the outbox. Models vary: if it doesn't call the tool, the cell says so; rerun it.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `RuntimeError: Cannot use Command(resume=...) without checkpointer` | Agent built without `checkpointer=` | Pass `InMemorySaver()` (or SQLite for real use) |
| Resume runs from scratch / nothing happens | Different `thread_id` in the resume config | Reuse the exact `cfg` of the paused run |
| `ValueError: Number of human decisions (1) does not match …` | One decision per paused call is required | Count `action_requests`; validate with `check_decisions` |
| Same error even after fixing the decisions | The failed resume is stuck on that thread | Start the request again on a new `thread_id` |
| `Decision type 'edit' is not allowed …` | Not in that tool's `allowed_decisions` | Use an allowed type or change the policy |
| `when=` raises `TypeError` / is ignored | langchain older than 1.3.3 | `pip install -e ".[dev]"` from the repo root to upgrade |
| The tool ran without a pause | Tool name in `interrupt_on` misspelled, or `when` returned `False` | Print `agent.get_graph()` or test the predicate alone |

## ⚠️ Production notes

> ⚠️ **Production note: approval is only as good as the review screen.**
> Show the **exact** arguments that will run (recipient, amount, full body), not a model summary of them; highlight what is risky (`description=`). Pause only for risky calls (`when=`) so reviewers stay attentive. Log who decided what and when (your audit middleware from 04.03), and set an expiry: a pause nobody answers should be rejected automatically, not approved.

> ⚠️ **Production note: HITL needs durable storage.**
> Use a persistent checkpointer (Postgres/SQLite, Module 03) with the same `thread_id` the UI holds. An in-memory saver loses every pending approval on restart or across workers.

## 🏋️ Exercises

1. **(Easy)** Make `ask_user` allow only `respond`, and `send_reply` only `approve`/`reject`, in one policy. Trigger both in one turn and resume with two decisions.
2. **(Medium)** Write `auto_review(pause_value) -> list[dict]` that rejects any reply to an email whose sender domain is not in an allow-list and approves the rest. Use it on e-105 + e-107.
3. **(Medium)** Change the `when=` rule so it also pauses for replies longer than 400 characters. Test both branches.
4. **(Stretch)** Build a tiny review queue: three threads paused in SQLite; list each pending interrupt with `get_state`, then approve them in a loop.

<details>
<summary><b>Show solutions</b></summary>

**1.** `{"ask_user": {"allowed_decisions": ["respond"]}, "send_reply": {"allowed_decisions": ["approve", "reject"]}}`; decisions in the order of `action_requests`, e.g. `[{"type": "respond", "message": "Formal"}, {"type": "approve"}]`.

**2.** For each request, look up `EMAILS[req["args"]["email_id"]]["from"]`, take the domain after `@`, and return `{"type": "reject", "message": "untrusted sender"}` or `{"type": "approve"}`. e-107 (`exfil.example`/`secure-mail.example`) is rejected.

**3.** `lambda r: bool(MONEY.search(b := r.tool_call["args"].get("body", ""))) or len(b) > 400`.

**4.** Keep the thread ids in a list; `[(t, agent.get_state(cfg(t)).interrupts) for t in threads]`; resume each with its own config.
</details>

## ✅ Checkpoint

- [ ] I can pause before a sensitive tool and read the interrupt (`version="v2"`, `.interrupts`).
- [ ] I can resume with approve, edit, reject and respond, and with several decisions in order.
- [ ] I can write a policy with `allowed_decisions`, `description` and `when=`.
- [ ] I can explain why HITL needs a checkpointer, and resumed a pause after "restarting" with SQLite.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

On the [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) page, find how decisions are sent when you **stream** instead of `invoke`. Note the pattern and one case where you'd stream a HITL agent, in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `HumanInTheLoopMiddleware` pauses after the model proposes and before tools run; nothing executes until you decide.
- Resume on the same thread with `Command(resume={"decisions": [...]})`: one decision per call, in order.
- Policies: `True`/`False`, or `allowed_decisions` + `description` + `when=`.
- Pauses live in the checkpointer; use a durable one so approvals survive restarts.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 04 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)